# PTB-XL: 1D CNN vs 1D ResNet on PTB-XL (five superclasses)

Runs the whole experiment on a Kaggle GPU: install the package from GitHub, get PTB-XL 1.0.3, prepare the 100 Hz signals with the official patient-level folds (1–8 train, 9 validation, 10 test), train both models with five seeds, and compare them with patient-level bootstrap intervals and selective-prediction curves.

**Settings:** Accelerator **GPU T4 x2** (or P100) and **Internet on**. Outputs are written to `/kaggle/working/reports`.

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/ptbxl-ecg-superclass-benchmark.git", "/kaggle/working/ptbxl-bench"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/ptbxl-bench"], check=True)
import os
os.chdir("/kaggle/working/ptbxl-bench")
# The kernel started before the install, so add the package source to the path directly.
sys.path.insert(0, "/kaggle/working/ptbxl-bench/src")
import cardionet
print("cardionet", cardionet.__version__)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

## 1. PTB-XL 1.0.3
Uses an attached Kaggle copy if one is found under `/kaggle/input`; otherwise downloads the two metadata files and the 100 Hz records from PhysioNet (takes a while the first time).

In [ ]:
import glob, urllib.request, zipfile, pathlib
BASE = "https://physionet.org/files/ptb-xl/1.0.3/"
found = glob.glob("/kaggle/input/**/ptbxl_database.csv", recursive=True)
if found:
    DATA_DIR = str(pathlib.Path(found[0]).parent)
else:
    # Official PhysioNet copy: the two metadata files plus only the 100 Hz records.
    DATA_DIR = "/kaggle/working/ptbxl"
    os.makedirs(DATA_DIR, exist_ok=True)
    for name in ["ptbxl_database.csv", "scp_statements.csv"]:
        urllib.request.urlretrieve(BASE + name, os.path.join(DATA_DIR, name))
    import pandas as pd
    from concurrent.futures import ThreadPoolExecutor

    def fetch(relative):
        target = os.path.join(DATA_DIR, relative)
        if not os.path.exists(target):
            os.makedirs(os.path.dirname(target), exist_ok=True)
            for attempt in range(5):
                try:
                    urllib.request.urlretrieve(BASE + relative, target)
                    break
                except Exception:
                    if attempt == 4:
                        raise

    records = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"))["filename_lr"].tolist()
    files = [r + ext for r in records for ext in (".hea", ".dat")]
    with ThreadPoolExecutor(max_workers=32) as pool:
        list(pool.map(fetch, files))
    print("downloaded", len(files), "files")
import pandas as pd
db = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"))
print(DATA_DIR, "| records:", len(db), "| patients:", db.patient_id.nunique())

## 2. Prepare signals (official folds, training-only normalisation)

In [ ]:
from cardionet.data import prepare_ptbxl
import json
meta = prepare_ptbxl(DATA_DIR, "data/processed", sampling_rate=100)
print(json.dumps(meta, indent=2))

## 3. Train both models with five seeds
Early stopping on validation loss (patience 5, up to 40 epochs). Thresholds are chosen per label on the validation fold and then frozen for the test fold.

In [ ]:
import torch, time
from cardionet.train import run_training
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
SEEDS = [17, 23, 31, 47, 59]
results = {}
for model in ["cnn", "resnet"]:
    for seed in SEEDS:
        t0 = time.time()
        m = run_training("configs/default.yaml", model, epochs=40, checkpoint_dir="checkpoints", seed=seed)
        results[(model, seed)] = m["macro_auroc"]
        print(f"{model} seed {seed}: macro AUROC {m['macro_auroc']:.4f} ({time.time() - t0:.0f} s)")

## 4. Compare

In [ ]:
from cardionet.analysis import analyse
from IPython.display import Image, Markdown, display
analyse("checkpoints", "/kaggle/working/reports", "data/processed")
display(Markdown(open("/kaggle/working/reports/analysis.md").read()))
display(Image("/kaggle/working/reports/fig_auroc_selective.png"))

In [ ]:
import shutil
for f in glob.glob("checkpoints/*_metrics.json") + glob.glob("checkpoints/*_predictions.npz") + glob.glob("checkpoints/*_best.pt") + ["data/processed/dataset_metadata.json"]:
    shutil.copy(f, "/kaggle/working/reports/")
shutil.rmtree("data/processed", ignore_errors=True)
shutil.rmtree("/kaggle/working/ptbxl", ignore_errors=True)
print(sorted(os.listdir("/kaggle/working/reports")))